# Music Recommendation System — Analysis Notebook
Explore SVD decomposition, latent features, and recommendation results.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D

from normalization import z_score_normalize
from svd import svd, explained_variance_ratio
from projection import project_features, interpret_latent_features, similarity_matrix
from recommend import recommend_one_per_song, recommend_overall_top

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
DATA_PATH = '../data/spotify_songs.csv'
FEATURE_COLS = ['danceability','energy','speechiness','acousticness',
                'instrumentalness','liveness','valence','loudness','tempo']

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(f'Loaded {len(df)} songs')
df[FEATURE_COLS].head()

## 2. Normalize & SVD

In [ ]:
B_raw = df[FEATURE_COLS].to_numpy(dtype=float)
B_norm, mean, std = z_score_normalize(B_raw)

K = 5
B_k, Vk, sigma, U = project_features(B_norm, k=K)
evr = explained_variance_ratio(sigma)

print('Singular values:', sigma.round(2))
print('Variance explained per component:', (evr * 100).round(1), '%')
print(f'Total variance explained: {evr.sum()*100:.1f}%')

## 3. Singular Value Scree Plot

In [ ]:
U_full, sigma_full, Vt_full = svd(B_norm)
evr_full = explained_variance_ratio(sigma_full)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(sigma_full)+1), sigma_full, color='steelblue')
axes[0].set_xlabel('Component'); axes[0].set_ylabel('Singular Value')
axes[0].set_title('Scree Plot — Singular Values')

axes[1].plot(range(1, len(evr_full)+1), np.cumsum(evr_full)*100, 'o-', color='coral')
axes[1].axhline(90, color='gray', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Explained Variance'); axes[1].legend()

plt.tight_layout(); plt.show()

## 4. Latent Feature Interpretation

In [ ]:
interp = interpret_latent_features(Vk, FEATURE_COLS)

fig, axes = plt.subplots(1, K, figsize=(18, 4))
colors_pos = 'steelblue'
colors_neg = 'coral'

for i in range(K):
    weights = Vk[:, i]
    colors = [colors_pos if w >= 0 else colors_neg for w in weights]
    axes[i].barh(FEATURE_COLS, weights, color=colors)
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].set_title(f'Latent Feature {i+1}\n(σ={sigma[i]:.1f})', fontsize=10)
    axes[i].set_xlim(-1, 1)

plt.suptitle('Latent Feature Loadings (SVD Right Singular Vectors)', fontsize=13)
plt.tight_layout(); plt.show()

## 5. 3D Projection Visualization

In [ ]:
PLAYLIST_TITLES = ['Kill Bill', 'Saturn', 'I Hate U', 'Low', 'Gone Girl']
playlist_mask = df['track_name'].str.lower().isin([t.lower() for t in PLAYLIST_TITLES])
playlist_idx = df[playlist_mask].index.tolist()

fig = plt.figure(figsize=(14, 5))

for col, (xi, yi, zi, title) in enumerate([
    (0, 1, 2, 'Original Feature Space\n(Danceability / Energy / Acousticness)'),
    (0, 1, 2, 'Latent Feature Space\n(SVD Projection)'),
]):
    ax = fig.add_subplot(1, 2, col+1, projection='3d')

    data = B_norm if col == 0 else B_k
    ax.scatter(data[:, xi], data[:, yi], data[:, zi],
               c='gray', alpha=0.2, s=5, label='All songs')
    if playlist_idx:
        pd_data = data[playlist_idx]
        ax.scatter(pd_data[:, xi], pd_data[:, yi], pd_data[:, zi],
                   c='red', s=40, label='Playlist', zorder=5)

    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Songs in Original vs. Latent Feature Space', fontsize=13)
plt.tight_layout(); plt.show()

## 6. Generate Recommendations

In [ ]:
song_metadata = df.to_dict(orient='records')

if playlist_idx:
    print('=== One-Per-Song Recommendations ===')
    recs_ops = recommend_one_per_song(playlist_idx, B_k, song_metadata)
    for r in recs_ops:
        print(f"  {r['track_name']} — {r['artist_names']}  [sim={r['similarity_score']:.4f}]")

    print('\n=== Overall Top Recommendations ===')
    recs_top = recommend_overall_top(playlist_idx, B_k, song_metadata, n_recommendations=10)
    for r in recs_top:
        print(f"  {r['track_name']} — {r['artist_names']}  [sim={r['similarity_score']:.4f}]")
else:
    print('Update PLAYLIST_TITLES above with songs present in your dataset.')